## TL;DR — Plain English

**What this notebook does:**
Takes you from zero knowledge to a working mental model of AlphaFold 3 in 3 hours.

**By the end, you will understand:**
- What a protein is (amino acid sequence → 3D structure)
- Why predicting the 3D structure from sequence was hard for 60 years
- The complete AF3 architecture: what goes in, what comes out, what each component does
- Why triangle attention (not regular attention) is needed for 3D structure
- How diffusion generates structures from noise
- The three equations that define AF3 mathematically

**You do NOT need to know:** biology, chemistry, structural biology, or advanced math.
You need: Python loops and functions, basic matrix multiply intuition.

**After this notebook:** Go to `07/01_af3_architecture.ipynb` for implementation details.


# AlphaFold 3 — Zero to Hero
## Start Here If You Have No Background

This notebook assumes you know **basic Python** but nothing about proteins, biology, or structure prediction.
By the end you will understand the entire AlphaFold 3 problem from the ground up.

**Who this is for:** Anyone starting modules 07 or 10 who finds the other notebooks assume too much.

**Time:** ~3 hours | **No prerequisites beyond Python lists and loops**


---
## Chapter 1: What Is a Protein? (Biology in 10 Minutes)

A protein is a **string of amino acids** that folds into a specific 3D shape.
That shape determines what the protein DOES in your body.

```
Amino acid string: M-V-H-L-T-P-E-E-K-S-A-V-T...
       ↓ (folds spontaneously in milliseconds)
3D structure: [complex tangled shape]
       ↓
Function: carries oxygen (hemoglobin), catalyzes reactions, builds cells
```

**There are only 20 standard amino acids.** Each has a 1-letter code (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y).


In [ ]:
# The 20 amino acids — learn these, they appear everywhere in AF3 code
amino_acids = {
    'A': ('Alanine',       'nonpolar',  'small',     'CHₓ side chain'),
    'C': ('Cysteine',      'nonpolar',  'small',     'SH — forms disulfide bonds'),
    'D': ('Aspartate',     'charged-',  'small',     'COO⁻ — negative at pH 7'),
    'E': ('Glutamate',     'charged-',  'medium',    'COO⁻ — negative, longer'),
    'F': ('Phenylalanine', 'aromatic',  'large',     'benzyl ring'),
    'G': ('Glycine',       'nonpolar',  'tiny',      'no side chain — max flexibility'),
    'H': ('Histidine',     'aromatic',  'medium',    'pKa≈6, switches charge at pH 6-8'),
    'I': ('Isoleucine',    'nonpolar',  'large',     'branched hydrophobic'),
    'K': ('Lysine',        'charged+',  'long',      'NH₃⁺ — positive'),
    'L': ('Leucine',       'nonpolar',  'large',     'branched hydrophobic'),
    'M': ('Methionine',    'nonpolar',  'medium',    'thioether — start codon AUG'),
    'N': ('Asparagine',    'polar',     'small',     'amide — hydrogen bonds'),
    'P': ('Proline',       'nonpolar',  'rigid',     'ring — breaks helix'),
    'Q': ('Glutamine',     'polar',     'medium',    'amide — hydrogen bonds, longer'),
    'R': ('Arginine',      'charged+',  'long',      'guanidinium — strongest +charge'),
    'S': ('Serine',        'polar',     'small',     'OH — phosphorylation target'),
    'T': ('Threonine',     'polar',     'medium',    'OH — phosphorylation target'),
    'V': ('Valine',        'nonpolar',  'medium',    'branched hydrophobic'),
    'W': ('Tryptophan',    'aromatic',  'largest',   'indole ring — fluorescent'),
    'Y': ('Tyrosine',      'aromatic',  'large',     'phenol — tyrosine kinase target'),
}

# In AF3 code, amino acids are encoded as integers 0-19 + special tokens
AA_TO_INT = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
INT_TO_AA = {v: k for k, v in AA_TO_INT.items()}

print("The 20 amino acids and their properties:")
print(f"{'AA'} {'Name':<15} {'Class':<12} {'Size':<8} {'Key feature'}")
print("-" * 65)
for aa, (name, cls, size, feat) in amino_acids.items():
    print(f" {aa}  {name:<15} {cls:<12} {size:<8} {feat}")

print(f"
AA_TO_INT encoding used in PyTorch:")
print("  'A' → 0, 'C' → 1, 'D' → 2 ...")
print(f"  Full alphabet: {AA_TO_INT}")
print()

# Key insight: hydrophobic AAs go INSIDE (away from water)
#              charged/polar AAs go OUTSIDE (interact with water)
hydrophobic = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if cls in ('nonpolar', 'aromatic')]
charged     = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if 'charged' in cls or cls == 'polar']
print(f"Hydrophobic (prefer inside): {hydrophobic}")
print(f"Polar/Charged (prefer surface): {charged}")
print()
print("WHY THIS MATTERS FOR AF3:")
print("  Hydrophobic collapse = primary driving force of protein folding")
print("  AF3 learns these preferences from millions of known structures")

---
## Chapter 2: The Three Levels of Protein Structure

```
PRIMARY:   M-V-H-L-T-P-E-E-K...        (the amino acid sequence — 1D)
              ↓ hydrogen bonds within chain
SECONDARY: α-helices, β-sheets, loops   (local regular patterns — 2D)
              ↓ packing of secondary elements
TERTIARY:  complete 3D fold             (one chain — 3D)
              ↓ assembly of multiple chains
QUATERNARY: protein complex             (multiple chains — 3D)
```

**AF3 predicts ALL four levels simultaneously — including complexes with DNA, RNA, and small molecules.**


In [ ]:
# VISUALIZING SECONDARY STRUCTURE ELEMENTS
# You will see alpha-helix and beta-sheet everywhere in AF3 code

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Generate idealized alpha-helix Cα coordinates
# Alpha helix: 3.6 residues/turn, 1.5 Å rise/residue, radius 2.3 Å
def ideal_helix(n_residues=20):
    coords = []
    for i in range(n_residues):
        angle = i * 2 * np.pi / 3.6       # 360°/3.6 residues per turn
        x = 2.3 * np.cos(angle)
        y = 2.3 * np.sin(angle)
        z = i * 1.5                         # 1.5 Å rise per residue
        coords.append([x, y, z])
    return np.array(coords)

# Generate idealized beta-strand (extended, alternating up/down)
def ideal_strand(n_residues=10):
    coords = []
    for i in range(n_residues):
        x = i * 3.5                         # 3.5 Å per residue in extended conformation
        y = (i % 2) * 0.8 - 0.4            # alternating up/down
        z = 0.0
        coords.append([x, y, z])
    return np.array(coords)

helix = ideal_helix(20)
strand = ideal_strand(10)

fig = plt.figure(figsize=(14, 5))

# Helix plot
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(helix[:,0], helix[:,1], helix[:,2], 'b-o', linewidth=2, markersize=5)
ax1.set_title('α-Helix
(3.6 res/turn, 1.5Å rise)')
ax1.set_xlabel('X (Å)'); ax1.set_ylabel('Y (Å)'); ax1.set_zlabel('Z (Å)')
for i in range(0, len(helix)-4, 4):
    ax1.plot([helix[i,0], helix[i+4,0]],
             [helix[i,1], helix[i+4,1]],
             [helix[i,2], helix[i+4,2]], 'r--', alpha=0.3, linewidth=1)

# Strand plot
ax2 = fig.add_subplot(132)
ax2.plot(strand[:,0], strand[:,1], 'g-o', linewidth=2, markersize=8)
ax2.set_title('β-Strand
(3.5Å per residue, alternating)')
ax2.set_xlabel('Position along strand (Å)')
ax2.set_ylabel('Side chain direction')
ax2.grid(True, alpha=0.3)

# Ramachandran plot (backbone dihedral angles)
np.random.seed(42)
# Alpha helix: phi≈-57°, psi≈-47°
helix_phi = np.random.normal(-57, 8, 300)
helix_psi = np.random.normal(-47, 8, 300)
# Beta sheet: phi≈-119°, psi≈+113°
sheet_phi = np.random.normal(-119, 10, 200)
sheet_psi = np.random.normal(+113, 10, 200)
# Left-handed helix (rare): phi≈+57°, psi≈+47°
lhelix_phi = np.random.normal(+57, 12, 30)
lhelix_psi = np.random.normal(+47, 12, 30)

ax3 = fig.add_subplot(133)
ax3.scatter(helix_phi, helix_psi, s=5, alpha=0.6, color='blue', label='α-helix')
ax3.scatter(sheet_phi, sheet_psi, s=5, alpha=0.6, color='green', label='β-sheet')
ax3.scatter(lhelix_phi, lhelix_psi, s=5, alpha=0.6, color='red', label='L-helix (rare)')
ax3.set_xlim(-180, 180); ax3.set_ylim(-180, 180)
ax3.axhline(0, color='gray', alpha=0.3); ax3.axvline(0, color='gray', alpha=0.3)
ax3.set_xlabel('φ (phi) backbone torsion angle °')
ax3.set_ylabel('ψ (psi) backbone torsion angle °')
ax3.set_title('Ramachandran Plot
(allowed regions cluster for each SS type)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('secondary_structure.png', dpi=100)
plt.show()

print("KEY INSIGHT:")
print("  Every residue has two backbone torsion angles: φ (phi) and ψ (psi)")
print("  These two numbers determine secondary structure!")
print("  α-helix: φ≈-57°, ψ≈-47°  (narrow cluster in top-left)")
print("  β-sheet: φ≈-119°, ψ≈+113° (upper-left spread)")
print()
print("  AF3's FAPE loss works in rigid frame coordinates defined by (N, CA, C)")
print("  The Ramachandran plot validates that predicted φ,ψ are physically reasonable")

---
## Chapter 3: Why Was Structure Prediction Hard Before AlphaFold?

The **protein folding problem** was unsolved for 50 years (1972–2020). Here's why:

### The Scale of the Problem
```
A 100-residue protein with 200 rotatable bonds,
each with 3 energy minima = 3²⁰⁰ ≈ 10⁹⁵ possible conformations.

The universe has ~10⁸⁰ atoms.

Random search is IMPOSSIBLE — Levinthal's paradox (1969).
```

### How Proteins Actually Fold (Answer: Energy Landscape)
Proteins don't search randomly. They fold along a **funnel-shaped energy landscape** — most conformations are unstable, native state is uniquely stable.

### Why Previous Computational Methods Failed
1. **Physics-based:** molecular dynamics is accurate but takes weeks per microsecond — folding takes milliseconds
2. **Fragment assembly:** Rosetta stitches known fragments — fails on novel folds
3. **Homology modeling:** works only if a similar structure is known
4. **Contact prediction:** predicts residue pairs in contact — but not full 3D coordinates

### AlphaFold's Insight (2020-2024)
**Co-evolution tells you structure.** If two residues co-mutate across species, they are in spatial contact.
AF2 learned to extract this signal from millions of sequence alignments.
AF3 extended this to DNA, RNA, small molecules, and replaced MSA-processing with diffusion.


In [ ]:
# CO-EVOLUTION SIGNAL — The key insight behind AF3
# If residue i and residue j co-mutate (change together across species),
# they are in spatial contact in the 3D structure.

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulate a protein family: 500 homologous sequences, 60-residue protein
n_seqs = 500
n_res = 60

# Define contact pairs (ground truth from structure)
true_contacts = [(5, 50), (10, 45), (15, 40), (20, 35), (25, 30)]

# Create MSA: sequences with co-evolution signal at contact positions
msa = np.random.randint(0, 4, (n_seqs, n_res))  # random background
for i, j in true_contacts:
    # Correlated mutations: when position i changes, position j changes too
    for s in range(n_seqs):
        if msa[s, i] % 2 == 0:
            msa[s, j] = (msa[s, j] + 1) % 4  # correlated change

# Compute naive pairwise correlation (simplified mutual information)
def mutual_info(col_i, col_j, n_states=4):
    joint = np.zeros((n_states, n_states))
    for a, b in zip(col_i, col_j):
        joint[a, b] += 1
    joint /= joint.sum()
    p_i = joint.sum(axis=1)
    p_j = joint.sum(axis=0)
    mi = 0
    for a in range(n_states):
        for b in range(n_states):
            if joint[a, b] > 0 and p_i[a] > 0 and p_j[b] > 0:
                mi += joint[a, b] * np.log(joint[a, b] / (p_i[a] * p_j[b]))
    return mi

print("Computing co-evolutionary signal from MSA...")
mi_matrix = np.zeros((n_res, n_res))
for i in range(n_res):
    for j in range(i+1, n_res):
        mi = mutual_info(msa[:, i], msa[:, j])
        mi_matrix[i, j] = mi_matrix[j, i] = mi

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. MSA heatmap (first 50 sequences)
axes[0].imshow(msa[:50, :], aspect='auto', cmap='tab10', interpolation='none')
axes[0].set_title('Multiple Sequence Alignment
(500 homologous proteins, 60 residues each)')
axes[0].set_xlabel('Residue position')
axes[0].set_ylabel('Sequence (of 500 homologs)')
axes[0].set_yticks([])

# 2. Co-evolution matrix
axes[1].imshow(mi_matrix, cmap='hot', aspect='auto')
for i, j in true_contacts:
    axes[1].plot(j, i, 'c+', markersize=15, markeredgewidth=2, label='True contact' if i == 5 else '')
    axes[1].plot(i, j, 'c+', markersize=15, markeredgewidth=2)
axes[1].set_title('Co-Evolution Matrix (Mutual Information)
(bright = co-mutating residue pairs)')
axes[1].set_xlabel('Residue j')
axes[1].set_ylabel('Residue i')
axes[1].legend(fontsize=8)
plt.colorbar(axes[1].images[0], ax=axes[1], label='Mutual Information')

# 3. Top predicted contacts vs true contacts
mi_flat = [(mi_matrix[i, j], i, j) for i in range(n_res) for j in range(i+5, n_res)]
mi_flat.sort(reverse=True)
top_10 = [(i,j) for _, i, j in mi_flat[:10]]

axes[2].scatter([], [], c='blue', s=100, label='True contacts')
axes[2].scatter([], [], c='red', marker='x', s=100, label='Predicted (top MI pairs)')
for (i, j) in true_contacts:
    axes[2].scatter(j, i, c='blue', s=200, zorder=5)
for k, (i, j) in enumerate(top_10):
    color = 'green' if (i,j) in true_contacts else 'red'
    axes[2].scatter(j, i, c=color, marker='x', s=200, zorder=5, linewidths=3)
axes[2].set_xlim(-1, n_res); axes[2].set_ylim(-1, n_res)
axes[2].set_title('Top 10 Co-Evolution Predictions
(green=correct, red=false positive)')
axes[2].set_xlabel('Residue j'); axes[2].set_ylabel('Residue i')
axes[2].legend()

plt.tight_layout()
plt.savefig('coevolution.png', dpi=100)
plt.show()

correct = sum(1 for (i,j) in top_10 if (i,j) in true_contacts or (j,i) in true_contacts)
print(f"Top 10 MI predictions: {correct}/5 true contacts found")
print()
print("THIS IS THE FOUNDATION OF AF3:")
print("  1. Collect thousands of homologous sequences (MSA)")
print("  2. Compute pairwise co-evolution signal")
print("  3. Co-evolving pairs = residues in 3D contact")
print("  4. Contacts constrain the 3D fold")
print("  5. AF3 learns to extract this signal automatically via the Pairformer")
print()
print("HISTORICALLY: DCA (2011) → structural biology research labs contact prediction (2017) → AF2 (2020) → AF3 (2024)")

---
## Chapter 4: AlphaFold 3 — The Complete Architecture in One Picture

```
INPUT: protein sequence(s) + DNA/RNA/ligand (optional) + MSA
            │
            ▼
    ┌───────────────────┐
    │  INPUT EMBEDDER   │  converts sequence tokens to feature vectors
    └────────┬──────────┘
             │  (s = single representation, z = pair representation)
             ▼
    ┌───────────────────┐
    │  PAIRFORMER STACK │  48 blocks of triangle attention + MLP
    │  (the "brain")    │  updates both s and z iteratively
    └────────┬──────────┘
             │  ↑ RECYCLED 3 times (rerun Pairformer on own output)
             ▼
    ┌───────────────────┐
    │  DIFFUSION MODULE │  generates 3D coordinates by denoising
    │  (structure gen.) │  starts from random noise → real structure
    └────────┬──────────┘
             │
             ▼
OUTPUT: 3D coordinates + confidence scores (pLDDT, PAE, pTM)
```

**AF3 vs AF2 key differences:**
| Component | AlphaFold 2 | AlphaFold 3 |
|-----------|------------|------------|
| Structure generation | IPA + angle prediction | Diffusion module |
| Residue encoding | Evoformer (MSA + pair) | Pairformer (pair only) |
| Supported molecules | Protein only | Protein + DNA + RNA + ligand |
| Output | Backbone + CB atoms | All heavy atoms |
| Confidence | pLDDT only | pLDDT + PAE + pTM + ipTM |


In [ ]:
# AF3 DATA FLOW — Step by Step with Actual Tensor Shapes
# This traces a single protein (50 residues) through the entire pipeline

import torch
import torch.nn as nn

# ─── PARAMETERS (AF3 at full scale vs our tutorial scale) ───
# Full AF3:   N≤5120, c_s=384, c_z=128, n_heads=16, 48 blocks
# Tutorial:   N=50,   c_s=64,  c_z=32,  n_heads=4,  2 blocks
# All shapes below marked as (Tutorial / Full-Scale)

N = 50     # sequence length (residues)
c_s = 64   # single representation dim  (tutorial: 64 / AF3: 384)
c_z = 32   # pair representation dim    (tutorial: 32 / AF3: 128)

print("AF3 DATA FLOW — tensor shapes at each stage")
print("=" * 60)
print(f"Input sequence: {N} residues")
print()

# ── STEP 1: Input Embedder ──
# Each residue → embedding vector
single_repr = torch.randn(N, c_s)      # (N, c_s)
pair_repr   = torch.randn(N, N, c_z)  # (N, N, c_z)
print(f"STEP 1 — Input Embedder:")
print(f"  Input: integer token per residue → shape (N,) = ({N},)")
print(f"  Output single repr s: {single_repr.shape}   (N × c_s = {N} × {c_s})")
print(f"  Output pair repr z:   {pair_repr.shape}  (N × N × c_z = {N} × {N} × {c_z})")
print()

# ── STEP 2: Pairformer Block (simplified) ──
def pairformer_step_shapes(N, c_s, c_z, n_heads=4):
    # Show what happens INSIDE one Pairformer block.
    print(f"STEP 2 — Pairformer Block (1 of 48):")
    print(f"  Input:  s={{{N},{c_s}}},  z={{{N},{N},{c_z}}}")
    print()
    print(f"  ├─ Triangle Attention Starting Node:")
    print(f"  │    For each row i: attend over columns j,k where z[i,j] and z[i,k] interact")
    print(f"  │    Q,K,V: ({N},{N},{c_z//n_heads}) each head")
    print(f"  │    Attention: ({N},{N},{N}) — THE BOTTLENECK (N³ memory!)")
    print(f"  │    Output z update: ({N},{N},{c_z})")
    print()
    print(f"  ├─ Triangle Multiplicative Update (Outgoing):")
    print(f"  │    For each pair (i,k): combine all edges (i,j)×(j,k) for all j")
    print(f"  │    Avoids N³ by using outer product + gating: ({N},{N},{c_z})")
    print()
    print(f"  ├─ Pair-to-Single Attention:")
    print(f"  │    Each residue i attends to its row of pair repr z[i,:,:]")
    print(f"  │    Updates s: ({N},{c_s})")
    print()
    print(f"  └─ Output: s={{{N},{c_s}}},  z={{{N},{N},{c_z}}}  (same shape — stacks 48 times)")

pairformer_step_shapes(N, c_s, c_z)
print()

# ── STEP 3: Diffusion Module ──
print(f"STEP 3 — Diffusion Module:")
print(f"  Input: conditioning from Pairformer (s, z)")
print(f"  Process: iterative denoising over T=200 noise levels")
print(f"  At step t: x_noisy = x_t + noise  →  predict x_0")
print(f"  Output: atom coordinates x: ({N}, 3)  [or (N_atoms, 3) for all-atom]")
print()
print(f"STEP 4 — Recycling (3 iterations):")
print(f"  Use output coordinates → re-embed → feed BACK into Pairformer")
print(f"  Each recycle refines structure prediction")
print(f"  Memory: intermediate activations kept for 3× longer")
print()
print(f"STEP 5 — Confidence Heads:")
print(f"  pLDDT: per-residue confidence  →  shape ({N},)")
print(f"  PAE:   pairwise aligned error  →  shape ({N},{N})")
print(f"  pTM:   global TM-score estimate →  scalar")
print()
print(f"TOTAL MEMORY (full AF3, N=384):")
print(f"  Pair repr z: {384*384*128*4/1e9:.2f} GB just for one example!")
print(f"  That's why chunked attention and gradient checkpointing exist.")

---
## Chapter 5: The Pairformer — Why Triangles?

The core question: **Why does AF3 use "triangle attention"? Why not regular self-attention?**

The answer lies in the geometry of distances.

**The Triangle Inequality**: For any three residues i, j, k in 3D space:
```
d(i,k) ≤ d(i,j) + d(j,k)    (triangle inequality)
```

If the model knows d(i,j) and d(j,k), it should update d(i,k) to be consistent.
**Triangle attention is the neural network operation that enforces this constraint.**

```
Triangle Attention:
  For pair (i,k): aggregate information from ALL intermediate residues j

  Updated z[i,k] = f(z[i,j], z[j,k] for all j)

  Biological interpretation:
  "How far apart are residues i and k?"
  "Let me check all residues j that I know the distance to, and infer (i,k)"
```


In [ ]:
# TRIANGLE INEQUALITY — visualize why triangles matter for 3D structure

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Place 5 residues in 3D space (true structure)
true_positions = np.array([
    [0, 0, 0],    # residue 0
    [4, 0, 0],    # residue 1
    [4, 3, 0],    # residue 2
    [0, 3, 0],    # residue 3
    [2, 1.5, 2],  # residue 4 (above the plane)
])
n = len(true_positions)

# Compute all true distances
def dist(a, b):
    return np.linalg.norm(a - b)

true_dists = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        true_dists[i,j] = dist(true_positions[i], true_positions[j])

print("TRUE DISTANCES (Ångstroms):")
print("     " + "  ".join([f"R{i}" for i in range(n)]))
for i in range(n):
    row = [f"{true_dists[i,j]:5.1f}" for j in range(n)]
    print(f"R{i}: {' '.join(row)}")

# Demonstrate: if we know d(0,1) and d(1,2), what can we infer about d(0,2)?
d01 = true_dists[0,1]  # 4.0
d12 = true_dists[1,2]  # 3.0
d02 = true_dists[0,2]  # 5.0

print(f"
TRIANGLE CONSTRAINT EXAMPLE:")
print(f"  d(0,1) = {d01:.1f} Å")
print(f"  d(1,2) = {d12:.1f} Å")
print(f"  d(0,2) = {d02:.1f} Å (to predict)")
print()
print(f"  Triangle inequality: d(0,2) ≤ d(0,1) + d(1,2) = {d01+d12:.1f} Å")
print(f"  Triangle inequality: d(0,2) ≥ |d(0,1) - d(1,2)| = {abs(d01-d12):.1f} Å")
print(f"  So d(0,2) must be in [{abs(d01-d12):.1f}, {d01+d12:.1f}] Å")
print(f"  True d(0,2) = {d02:.1f} Å ← within this range ✓")
print()
print(f"WHAT PAIRFORMER DOES:")
print(f"  Triangle attention aggregates ALL intermediate residues j=0..N")
print(f"  to update the pair representation z[i,k] for target pair (i,k)")
print(f"  It learns to enforce triangle consistency automatically!")
print()

# Show violation: if we corrupt one distance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(true_positions[:, 0], true_positions[:, 1],
           s=200, c=range(5), cmap='viridis', zorder=5)
for i in range(n):
    for j in range(i+1, n):
        ax.plot([true_positions[i,0], true_positions[j,0]],
                [true_positions[i,1], true_positions[j,1]],
                'gray', alpha=0.4, linewidth=1)
        mid = (true_positions[i] + true_positions[j]) / 2
        ax.text(mid[0], mid[1], f'{true_dists[i,j]:.1f}', fontsize=7,
                ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
for i in range(n):
    ax.annotate(f'R{i}', true_positions[i,:2], textcoords='offset points',
                xytext=(5, 5), fontsize=11, fontweight='bold')
ax.set_title('5 Residues: All Pairwise Distances
(triangle inequality constrains each)')
ax.set_xlabel('X (Å)'); ax.set_ylabel('Y (Å)')

# Show triangle attention circuit
ax2 = axes[1]
i_node, k_node, j_nodes = 0, 2, [1, 3, 4]
ax2.scatter([true_positions[i_node, 0]], [true_positions[i_node, 1]],
            s=400, c='red', zorder=6, label='query (i)')
ax2.scatter([true_positions[k_node, 0]], [true_positions[k_node, 1]],
            s=400, c='blue', zorder=6, label='key (k)')
ax2.scatter([true_positions[j, 0] for j in j_nodes],
            [true_positions[j, 1] for j in j_nodes],
            s=200, c='green', zorder=6, label='intermediary j')
for j in j_nodes:
    ax2.annotate('', xy=true_positions[k_node, :2],
                 xytext=true_positions[j, :2],
                 arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
    ax2.annotate('', xy=true_positions[j, :2],
                 xytext=true_positions[i_node, :2],
                 arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
ax2.plot([true_positions[i_node,0], true_positions[k_node,0]],
         [true_positions[i_node,1], true_positions[k_node,1]],
         'r--', linewidth=3, label='target pair (i,k)')
ax2.set_title('Triangle Attention Circuit
Update z[i,k] using all intermediary j')
ax2.legend(fontsize=9)
ax2.set_xlabel('X (Å)'); ax2.set_ylabel('Y (Å)')

plt.tight_layout()
plt.savefig('triangle_intuition.png', dpi=100)
plt.show()

---
## Chapter 6: Diffusion in AF3 — Structure Generation as Denoising

**Key insight:** Instead of directly predicting coordinates (AF2's approach), AF3 *generates* structures by learning to remove noise.

```
FORWARD PROCESS (training):
  x_0  →  x_1  →  x_2  →  ...  →  x_T  (add more noise each step)
  (real structure)                    (pure Gaussian noise)

REVERSE PROCESS (inference):
  x_T  →  x_{T-1} →  ...  →  x_1  →  x_0  (predict and remove noise)
  (noise)                               (predicted structure)

At each reverse step t:
  x_0_pred = Model(x_t, t, conditioning)    ← Pairformer provides conditioning!
  x_{t-1} = DDPM update rule(x_t, x_0_pred)
```

**Why diffusion instead of direct coordinate prediction?**
1. **Multiple conformations:** diffusion naturally samples from a distribution — run it 5 times, get 5 different valid structures
2. **All-atom prediction:** easier to generate 3D point clouds than predict named atom positions
3. **No need for torsion angles:** AF2 predicted φ,ψ angles + IPA; AF3 predicts xyz directly


In [ ]:
# DIFFUSION CONCEPTS — From Scratch
# Understanding the forward and reverse processes

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── FORWARD PROCESS: adding noise ──
def forward_process(x0, t, T=200):
    # Add noise to structure x0 at timestep t.
    # AF3 uses a variance-preserving schedule.
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise
    # alpha_bar_t = product(1 - beta_s for s in 0..t)
    # Linear noise schedule (simplified; AF3 uses a different schedule)
    betas = torch.linspace(1e-4, 0.02, T)
    alpha_bars = torch.cumprod(1 - betas, dim=0)

    alpha_bar_t = alpha_bars[t]
    noise = torch.randn_like(x0)
    x_t = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * noise
    return x_t, noise

# Simulate a 20-residue protein backbone (just Cα)
N = 20
# Create a simple helix as "true structure"
t_vals = torch.arange(N).float()
x0 = torch.stack([
    2.3 * torch.cos(t_vals * 2 * np.pi / 3.6),
    2.3 * torch.sin(t_vals * 2 * np.pi / 3.6),
    t_vals * 1.5,
], dim=-1)  # shape (20, 3)

# Show forward process at different noise levels
T = 200
timesteps_to_show = [0, 25, 50, 100, 150, 199]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, t in zip(axes.flat, timesteps_to_show):
    x_t, _ = forward_process(x0, t, T)
    coords = x_t.numpy()

    ax.plot(coords[:, 0], coords[:, 1], 'b-o', linewidth=1.5, markersize=4, alpha=0.8)
    ax.set_xlim(-8, 8); ax.set_ylim(-6, 6)
    ax.set_title(f't = {t}/{T}
{"(clean structure)" if t==0 else "(pure noise)" if t==199 else ""}')
    ax.set_xlabel('X (Å)'); ax.set_ylabel('Y (Å)')

    # Compute SNR
    betas = torch.linspace(1e-4, 0.02, T)
    alpha_bars = torch.cumprod(1 - betas, dim=0)
    snr = alpha_bars[t] / (1 - alpha_bars[t])
    ax.set_title(f't = {t}/{T}  SNR = {snr:.3f}')
    ax.grid(True, alpha=0.2)

plt.suptitle('AF3 Diffusion: Forward Process (Adding Noise to Protein Structure)
'
             'Training: model learns to reverse this process at each timestep',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('diffusion_forward.png', dpi=100)
plt.show()

print("DIFFUSION KEY CONCEPTS:")
print()
print("  FORWARD (training data generation):")
print("    x_0 = real structure from PDB")
print("    x_t = x_0 + Gaussian noise (more noise as t increases)")
print("    At t=T: x_T ≈ pure noise (no structure information)")
print()
print("  REVERSE (inference — generating new structures):")
print("    Start from x_T ~ N(0, I)")
print("    Model predicts x_0 from x_t and timestep t")
print("    Update to x_{t-1} using DDPM rule")
print("    After T steps: x_0 = predicted structure")
print()
print("  CONDITIONING in AF3:")
print("    At every reverse step, the Pairformer output (s, z) is fed to the model")
print("    This is how sequence information guides structure generation")
print("    Without conditioning: random protein backbone (like DDPM for images)")
print("    With AF3 conditioning: protein-specific fold")
print()
print("  WHY SAMPLE 5 TIMES?")
print("    Because the reverse process is stochastic — different noise → different structure")
print("    For intrinsically disordered proteins: 5 samples show DIFFERENT conformations")
print("    For well-folded proteins: 5 samples are nearly identical (→ low pLDDT variance)")

---
## Chapter 7: Study Path from Zero to AF3 Engineer

Now that you understand the concepts, here is the exact study path through the notebooks.

### Prerequisite Check
Run this cell to check if you're ready for the AF3 notebooks.


In [ ]:
# PREREQUISITE CHECKER — honest assessment before diving in
print("AF3 PREREQUISITE SELF-ASSESSMENT")
print("=" * 50)
print("Rate yourself honestly 1-5 on each:")
print()

prerequisites = [
    ("Transformers/Attention",
     "Can you explain multi-head attention and write it from scratch?",
     "Module 05/01 — attend to the MHA section before proceeding"),

    ("PyTorch nn.Module",
     "Can you write a custom nn.Module with forward() and use it in a training loop?",
     "Module 00/06 — PyTorch fundamentals"),

    ("Matrix multiplication intuition",
     "If X has shape (N,d) and W has shape (d,k), what is the shape of X@W?",
     "Module 00/08 — mathematical foundations"),

    ("Protein basics",
     "Can you explain what amino acids are and why protein shape matters?",
     "This notebook Chapter 1 — you just read it!"),

    ("3D coordinate geometry",
     "If you rotate coordinates by a rotation matrix R, what is R@x?",
     "Module 07/01 will introduce rigid frames — RMSD in Module 03 helps first"),

    ("Gradient descent",
     "Can you explain what happens when you call loss.backward() in PyTorch?",
     "Module 00/06 — autograd section"),

    ("Loss functions",
     "What is the difference between MSE loss and cross-entropy loss?",
     "Module 00/02 — ML fundamentals"),
]

for topic, question, if_no in prerequisites:
    print(f"  {topic}:")
    print(f"    Question: {question}")
    print(f"    If not ready: {if_no}")
    print()

print("RECOMMENDED STUDY ORDER:")
print()
print("WEEK 1 (prerequisites):")
print("  Day 1-2: Module 00/06 (PyTorch) + 00/08 (Math)")
print("  Day 3-4: Module 05/01 (Attention, Transformers)")
print("  Day 5-7: Module 03/01 (Structural Biology basics)")
print()
print("WEEK 2 (AF3 core):")
print("  Day 1:   THIS NOTEBOOK (07/00 — concepts and intuition)")
print("  Day 2-3: Module 07/01 (AF3 architecture — rigid frames, FAPE, triangle attention)")
print("  Day 4-5: Module 07/02 (AF3 data pipeline — mmCIF, MSA, tokenization)")
print("  Day 6-7: Module 07/03 (AF3 evaluation — pLDDT, PAE, DockQ)")
print()
print("WEEK 3 (advanced + practical):")
print("  Day 1-2: Module 07/04 (full-scale AF3 — chunked attention, CCD)")
print("  Day 3-5: Module 10/00 (OpenFold code walkthrough)")
print("  Day 6-7: Module 10/01 (fine-tuning capstone)")
print()
print("AFTER COMPLETING:")
print("  You will be able to: read AF3/Boltz-2 source code, implement components from")
print("  scratch, fine-tune for new tasks, and evaluate predictions critically.")
print("  Target: attempt the CASP15 evaluation exercise in Module 07/03.")

---
## Chapter 8: The Three Most Important Equations in AF3

Before touching any code in the other notebooks, understand these three equations. They appear everywhere.

### Equation 1: Scaled Dot-Product Attention
$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**In English:** For each query vector, compute similarity to all keys. Normalize with softmax. Take a weighted sum of values.

### Equation 2: FAPE Loss (Frame Aligned Point Error)
$$\text{FAPE} = \frac{1}{N_\text{frames}} \sum_i \frac{1}{N_\text{atoms}} \sum_j \min(\|\hat{R}_i^T(\hat{x}_j - \hat{t}_i) - R_i^T(x_j - t_i)\|, d_\text{clamp})$$

**In English:** For each predicted rigid frame $\hat{R}_i, \hat{t}_i$: transform all atoms into that local frame. Compare predicted vs. true local coordinates. Average over all frames.

### Equation 3: Diffusion Reverse Step (DDPM)
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\hat{\epsilon}_\theta(x_t, t)\right) + \sigma_t z$$

**In English:** At each denoising step: remove the predicted noise $\hat{\epsilon}$ (scaled), then add a small amount of fresh noise $z$ (this keeps the sampling stochastic).


In [ ]:
# THE THREE EQUATIONS — verify your understanding with numerical examples

import torch
import numpy as np

print("EQUATION 1: Scaled Dot-Product Attention")
print("-" * 50)

# Example: N=4 tokens, d_k=8
N, d_k = 4, 8
torch.manual_seed(42)
Q = torch.randn(N, d_k)  # queries
K = torch.randn(N, d_k)  # keys
V = torch.randn(N, d_k)  # values

# Compute attention
scores = Q @ K.T / (d_k ** 0.5)   # (N, N)
attn_weights = torch.softmax(scores, dim=-1)  # (N, N), rows sum to 1
output = attn_weights @ V           # (N, d_k)

print(f"  Q shape: {Q.shape}")
print(f"  K shape: {K.shape}")
print(f"  Scores (QK^T / sqrt(dk)): {scores.shape}, range=[{scores.min():.2f}, {scores.max():.2f}]")
print(f"  Attention weights (softmax): {attn_weights.shape}")
print(f"  Each row sums to 1: {attn_weights.sum(dim=-1).tolist()}")
print(f"  Output: {output.shape}")
print()
print("  WHY divide by sqrt(d_k)?")
print("    QK^T has variance ≈ d_k (sum of d_k products)")
print("    Dividing by sqrt(d_k) normalizes to variance ≈ 1")
print("    Without this: softmax saturates → vanishing gradients")
print()

print("EQUATION 2: FAPE Loss")
print("-" * 50)

def make_rotation(angle_x, angle_y, angle_z):
    # Simple 3D rotation matrix (Rx @ Ry)
    Rx = torch.tensor([[1, 0, 0],
                        [0, np.cos(angle_x), -np.sin(angle_x)],
                        [0, np.sin(angle_x),  np.cos(angle_x)]])
    Ry = torch.tensor([[np.cos(angle_y), 0, np.sin(angle_y)],
                        [0, 1, 0],
                        [-np.sin(angle_y), 0, np.cos(angle_y)]])
    return (Rx @ Ry).float()

# 5 residues, each defining a local frame
n_res = 5
n_atoms = 10
d_clamp = 10.0

# True coordinates
true_coords = torch.randn(n_atoms, 3)

# True frames (rotations + translations)
true_R = torch.stack([make_rotation(0.1*i, 0.05*i, 0.02*i) for i in range(n_res)])
true_t = torch.randn(n_res, 3)

# Predicted (slightly wrong)
pred_coords = true_coords + torch.randn_like(true_coords) * 0.5  # add noise
pred_R = torch.stack([make_rotation(0.1*i+0.05, 0.05*i+0.02, 0.02*i+0.01) for i in range(n_res)])
pred_t = true_t + torch.randn_like(true_t) * 0.2

# Compute FAPE
fape_total = 0
for i in range(n_res):
    for j in range(n_atoms):
        # Transform atom j into frame i
        true_local  = true_R[i].T @ (true_coords[j]  - true_t[i])
        pred_local  = pred_R[i].T @ (pred_coords[j]  - pred_t[i])
        dist = (true_local - pred_local).norm()
        fape_total += min(dist.item(), d_clamp)

fape = fape_total / (n_res * n_atoms)
print(f"  {n_res} frames × {n_atoms} atoms = {n_res*n_atoms} local error terms")
print(f"  FAPE = {fape:.4f} Å (smaller = better prediction)")
print(f"  FAPE = 0 means perfect prediction in every local frame")
print()
print("  WHY use local frames instead of global RMSD?")
print("    Global RMSD is sensitive to rigid-body rotation (same structure, different orientation)")
print("    Local frame evaluation is INVARIANT to rigid-body rotation")
print("    Also: d_clamp=10Å prevents single bad prediction from dominating loss")
print()

print("EQUATION 3: DDPM Reverse Step")
print("-" * 50)

T = 200
t_step = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# At step t=100, show the update
alpha_t = alphas[t_step]
alpha_bar_t = alpha_bars[t_step]
sigma_t = torch.sqrt(betas[t_step])

x_t = torch.randn(5, 3)  # noisy coordinates
eps_pred = torch.randn(5, 3)  # model's noise prediction

x_t_minus_1 = (1 / torch.sqrt(alpha_t)) * (
    x_t - (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t) * eps_pred
) + sigma_t * torch.randn_like(x_t)

print(f"  At timestep t={t_step}/{T}:")
print(f"    alpha_t     = {alpha_t:.6f}  (1 - beta_t, close to 1)")
print(f"    alpha_bar_t = {alpha_bar_t:.6f}  (cumulative, drops with t)")
print(f"    sigma_t     = {sigma_t:.6f}  (noise std at this step)")
print(f"    |x_t - x_{{t-1}}| = {(x_t - x_t_minus_1).norm():.4f}  (small step)")
print()
print("  At t=T (pure noise): alpha_bar_T ≈ 0 → all information is noise")
print("  At t=0 (clean):      alpha_bar_0 = 1 → no noise")
print("  Each step removes a tiny bit of noise from the protein coordinates")

---
## Summary: What You Now Know

You started with no background. You now understand:

1. **Proteins**: amino acid sequences that fold into 3D structures determining function
2. **The folding problem**: was unsolved for 50 years; requires predicting 3D structure from sequence
3. **Co-evolution**: MSAs reveal contact information; AF3 learns this via the Pairformer
4. **AF3 architecture**: Embedder → Pairformer (48 blocks) → Diffusion → Confidence heads
5. **Triangle attention**: enforces 3D geometric consistency (triangle inequality) in the pair representation
6. **Diffusion**: structure generation by learning to denoise; enables multiple conformation sampling
7. **The three key equations**: scaled dot-product attention, FAPE loss, DDPM reverse step

## Next Notebooks (in order)
1. **07/01 — AF3 Architecture**: implement every component from scratch
2. **07/02 — AF3 Data Pipeline**: mmCIF parsing, MSA featurization, tokenization
3. **07/03 — AF3 Evaluation**: pLDDT, PAE, DockQ — evaluate real predictions
4. **07/04 — Full-Scale AF3**: chunked attention, gradient checkpointing, CCD
5. **10/00 — OpenFold Code Walkthrough**: navigate real source code
6. **10/01 — Fine-Tuning Capstone**: LoRA on Pairformer for ΔΔG prediction


---
## 📚 Resources — Learning AlphaFold 3 from Zero

### Start Here If You Have No Biology Background
| Resource | Time | Why |
|----------|------|-----|
| **[Khan Academy Proteins](https://www.khanacademy.org/science/ap-biology/gene-expression-and-regulation/translation/a/amino-acids-article)** | 30 min | 20 amino acids, protein structure hierarchy |
| **[iBiology: How do proteins fold?](https://www.ibiology.org/biochemistry/protein-folding/)** | 1 hour | Visual introduction; free |
| **[3Blue1Brown: Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)** | 3 hours | Attention, backprop from first principles |
| **[Jay Alammar: The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)** | 1 hour | Best visual attention explanation |

### Start Here If You Have Biology Background but Not ML
| Resource | Time | Why |
|----------|------|-----|
| **[fast.ai Practical Deep Learning Part 1](https://course.fast.ai/)** | 20 hours | Best coding-first intro; free |
| **[MIT 6.S191 Deep Learning](https://introtodeeplearning.com/)** | 12 hours | Official MIT course; excellent video quality |
| **[Stanford CS229 Notes Ch 1-5](https://cs229.stanford.edu/lectures/)** | 6 hours | Mathematical foundations; free PDF |

### The AlphaFold Learning Sequence (University-Recommended)
```
Week 1: This notebook (07/00) + MIT 6.S191 Lab 1
Week 2: 07/01 Architecture + Jumper's EMBL talk (1hr YouTube)
Week 3: 07/02 Data + read AF3 Methods Section 2
Week 4: 07/03 Evaluation + implement TM-score from scratch
Week 5: 07/04 Full scale + Flash Attention paper
Week 6: 07/05 Training loop + OpenFold loss.py code reading
Week 7: 10/00 OpenFold walkthrough + clone Boltz repo
Week 8: 10/01 Fine-tuning capstone
```

### The Three Must-Read Papers (with free PDFs)
1. **[AlphaFold 2 (Nature 2021)](https://www.nature.com/articles/s41586-021-03819-2)** — Read abstract + Results + the supplement Algorithm boxes
2. **[AlphaFold 3 (Nature 2024)](https://www.nature.com/articles/s41586-024-07487-w)** — Read Methods Sections 1-3; rest is advanced
3. **[Boltz-1 Technical Report](https://www.biorxiv.org/content/10.1101/2024.11.19.624167v1)** — More accessible AF3 description; open-weight

### MIT/Harvard Structural Biology Context
- **[MIT 5.069/7.72 Crystal Structure Analysis](https://ocw.mit.edu/)** — X-ray crystallography context for PDB structures
- **[Harvard BBS 232 Structural Biology](https://gsas.harvard.edu/)** — Advanced; how experimental structures are determined
- **[MIT 6.874 Computational Biology of Disease](https://mit6874.github.io/)** — Connects AF3 to drug discovery directly


---
## 🎯 Interview Questions — AlphaFold 3 Concepts (Entry Level)

These questions are asked when you have 0-1 years of AF3 experience.

**Q1 (Conceptual):** What problem does AlphaFold 3 solve, in one sentence?
> **A:** AF3 predicts the 3D atomic structure of proteins (and their complexes with DNA, RNA, and small molecules) from their amino acid sequence.

**Q2 (Conceptual):** Why couldn't we solve protein folding before 2020?
> **A:** The search space is astronomically large (Levinthal's paradox: 10³⁰⁰ possible conformations). Classical physics-based methods couldn't search it fast enough. AF2 succeeded by learning from the 200,000+ known structures in the PDB, plus co-evolution signals from millions of homologous sequences.

**Q3 (Key difference):** What is the main architectural difference between AF2 and AF3?
> **A:** AF2 uses Evoformer (updates both MSA + pair representations) + IPA structure module (predicts backbone angles). AF3 uses Pairformer (updates only pair + single representations, more efficient) + diffusion module (generates 3D coordinates by denoising from Gaussian noise, enables all-atom prediction including ligands).

**Q4 (Understanding pLDDT):** A protein prediction has pLDDT = 40 for residues 100-150. What does this mean?
> **A:** pLDDT < 50 means the model is not confident in those residues' positions. This typically indicates an intrinsically disordered region (IDR) — a segment that doesn't adopt a fixed 3D structure in solution. Don't trust the predicted coordinates; the sequence may still be functional but structurally flexible.

**Q5 (Practical):** You submit a protein to the AF3 server and get pTM = 0.3. Should you trust the prediction?
> **A:** pTM < 0.5 indicates low confidence in the overall fold topology. For multi-domain proteins or novel folds, this is common. Check the PAE (Predicted Aligned Error) plot — if off-diagonal blocks are also high, the domain arrangement is uncertain. For drug design, you'd want pTM > 0.7 and pLDDT > 70 in the region of interest.


---
## 🔧 Debug Exercises — AlphaFold 3 Concepts

These are the mistakes beginners make when first implementing AF3 components.


In [ ]:
# DEBUG EXERCISE: Broken attention implementation
# Can you find the 3 bugs in this scaled dot-product attention?

import torch
import torch.nn.functional as F

def attention_buggy(Q, K, V):
    # Q, K, V: (batch, seq_len, d_k)
    d_k = Q.shape[-1]

    # Bug 1: wrong scaling
    scores = Q @ K                          # BUG: should be Q @ K.transpose(-2,-1) / sqrt(d_k)

    # Bug 2: wrong softmax dimension
    attn_weights = F.softmax(scores, dim=0) # BUG: should be dim=-1 (over keys, not batch)

    # Bug 3: wrong output
    output = attn_weights @ Q               # BUG: should multiply by V, not Q

    return output

def attention_correct(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)  # Fix 1: transpose K, scale by sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)           # Fix 2: softmax over key dimension
    output = attn_weights @ V                          # Fix 3: weight the values
    return output

torch.manual_seed(42)
B, L, d_k = 2, 5, 8
Q = torch.randn(B, L, d_k)
K = torch.randn(B, L, d_k)
V = torch.randn(B, L, d_k)

try:
    out_buggy = attention_buggy(Q, K, V)
    print(f"Buggy output shape: {out_buggy.shape}")
    print(f"Expected: ({B}, {L}, {d_k}) but got: {out_buggy.shape}")
except Exception as e:
    print(f"Buggy version error: {e}")

out_correct = attention_correct(Q, K, V)
print(f"Correct output shape: {out_correct.shape}  (should be {(B, L, d_k)})")
print()
print("LESSON: The 3 most common attention bugs:")
print("  1. Forgetting to transpose K (need K^T for dot product with Q)")
print("  2. Softmax on wrong dimension (must be over keys, i.e. dim=-1)")
print("  3. Multiplying attention by Q or K instead of V")

## ✅ Mastery Check — AlphaFold3: Zero to Hero

_Answer these questions to verify your understanding._

**Q1 (Recall):** What are the four main input representations that AlphaFold3 uses? What is the role of the MSA (Multiple Sequence Alignment) in structure prediction?

**Q2 (Understand):** Explain in plain English what the Pairformer does. Why does AF3 use a pair representation in addition to a single sequence representation?

**Q3 (Apply):** FAPE loss is used instead of standard coordinate RMSD as the training loss. What property of FAPE makes it better suited for training protein structure models? Give a concrete example of when RMSD would fail but FAPE would not.

**Q4 (Analyze):** AF3 replaced AF2's IPA (Invariant Point Attention) with a diffusion-based structure module. What are the theoretical advantages of diffusion for structure generation compared to regression-based approaches?

**Q5 (Design):** You want to fine-tune AF3 to predict antibody-antigen complex structures. Which components of AF3 would you fine-tune, which would you freeze, and what training data would you curate?

---
**Self-assessment rubric:**
- Q1–Q3: ready to move to Module 07/01 (AF3 Architecture deep dive)
- Q1–Q4: strong understanding of the AF3 design decisions
- Q1–Q5: interview-ready for computational biology ML teams / structural biology research labs ML engineer roles


---
## 🌍 Real-World Context — Where This Knowledge Is Used Today

This zero-to-hero overview prepares you for the exact tools used at the frontier of drug discovery in 2024-2025.

### Companies Using AF3-Level Technology Right Now

| Company | Tool | What They Do With It |
|---------|------|----------------------|
| **computational biology ML teams** | AF3 + internal models | De novo drug design; protein-ligand docking at scale |
| **drug discovery companies** | Chroma (diffusion) | Design antibodies against cancer targets |
| **drug discovery companies Pharma** | ESM-2 + structure models | Screen 100,000 compounds/week using ML |
| **Eikon Therapeutics** | AF2/3 + single-molecule imaging | Resolve protein conformations drug companies missed |
| **Relay Therapeutics** | MD + ML hybrids | Design drugs that work on protein conformational dynamics |

### What "AF3 Engineer" Actually Means Day-to-Day

1. **Run predictions:** Submit sequences to AF3 server, parse JSON confidence outputs, flag low-pLDDT regions
2. **Fine-tune:** Adapt AF3/OpenFold on proprietary data (protein families, membrane proteins, protein-ligand complexes)
3. **Debug:** Identify why a prediction has low confidence; redesign the experiment or sequence
4. **Integrate:** Connect AF3 outputs to docking pipelines, ΔΔG models, generative design loops
5. **Scale:** Run 10,000 predictions per week; build automated pipelines with error handling

### The 5 Things That Make You Hireable

After completing Modules 07 and 10, you'll be able to demonstrate all five in an interview:
1. Explain Pairformer architecture in 2 minutes to a non-ML scientist
2. Implement FAPE loss from scratch (Module 07/05)
3. Fine-tune OpenFold on a custom protein family (Module 10/01)
4. Interpret pLDDT and PAE from a real AF3 output (Module 07/03)
5. Connect structure prediction to downstream drug discovery (Module 17 capstone)
